# 02. Graph API 기초 — StateGraph로 워크플로 만들기

## 학습 목표

StateGraph, 노드, 엣지, 조건부 분기, 상태 리듀서를 이해합니다.

## 2.1 환경 설정

LLM 모델과 필요한 모듈을 불러옵니다.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 2.2 StateGraph 기본 구조

StateGraph를 사용하는 기본 흐름은 다음과 같습니다:

1. `StateGraph(State)` — 상태 스키마로 그래프 빌더 생성
2. `add_node()` — 노드(함수) 등록
3. `add_edge()` — 노드 간 연결
4. `compile()` — 실행 가능한 그래프 생성
5. `invoke()` — 그래프 실행

```
StateGraph(State) → add_node() → add_edge() → compile() → invoke()
```

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


class State(TypedDict):
    text: str
    word_count: int


def count_words(state: State) -> dict:
    words = state["text"].split()
    return {
        "word_count": len(words)
    }


builder = StateGraph(State)

builder.add_node("counter", count_words)

builder.add_edge(START, "counter")
builder.add_edge("counter", END)

graph = builder.compile()


result = graph.invoke(
    {
        "text": "LangGraph는 에이전트를 구축하기 위한 강력한 프레임워크입니다."
    },
    config=lf_config
)

print(f"텍스트: {result['text']}")
print(f"단어 수: {result['word_count']}")

텍스트: LangGraph는 에이전트를 구축하기 위한 강력한 프레임워크입니다.
단어 수: 6


## 2.3 상태 리듀서

`Annotated`로 상태 업데이트 방식을 지정합니다.

### 리듀서란?

- 리듀서는 상태의 필드가 **어떻게 업데이트되는지** 결정합니다
- 리듀서 없음: 단순 덮어쓰기 (override)
- `operator.add`: 리스트 항목을 누적 (append)
- 커스텀 함수로 리듀서 정의도 가능

In [4]:
from typing import Annotated, TypedDict
import operator

class AccState(TypedDict):
    items: Annotated[list[str], operator.add]  # list append via reducer
    count: int  # default: override


def step_a(state: AccState) -> dict:
    return {
        "items": ["from_a"],
        "count": 1
    }


def step_b(state: AccState) -> dict:
    return {
        "items": ["from_b"],
        "count": 2
    }


builder = StateGraph(AccState)

builder.add_node("a", step_a)
builder.add_node("b", step_b)

builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)

graph = builder.compile()

result = graph.invoke(
    {
        "items": [],
        "count": 0
    },
    config=lf_config
)

print(f"items (리듀서=add): {result['items']}")
# ["from_a", "from_b"]

print(f"count (덮어쓰기): {result['count']}")
# 2

items (리듀서=add): ['from_a', 'from_b']
count (덮어쓰기): 2


## 2.4 조건부 엣지

상태에 따라 다른 노드로 분기합니다.

- `add_conditional_edges(source, routing_function)` — 라우팅 함수의 반환값이 다음 노드 이름
- 라우팅 함수는 `Literal` 타입 힌트를 사용하면 시각화에 도움

```
START → classify → [route] → weather → END
                           → math    → END
                           → general → END
```

In [5]:
from typing import Literal

class RouterState(TypedDict):
    query: str
    category: str
    result: str

def classify(state: RouterState) -> dict:
    q = state["query"].lower()
    if "weather" in q or "날씨" in q:
        return {"category": "weather"}
    elif "math" in q or "calculate" in q or "계산" in q:
        return {"category": "math"}
    return {"category": "general"}

def handle_weather(state: RouterState) -> dict:
    return {"result": f"[날씨] 날씨 확인 중: {state['query']}"}

def handle_math(state: RouterState) -> dict:
    return {"result": f"[수학] 계산 중: {state['query']}"}

def handle_general(state: RouterState) -> dict:
    return {"result": f"[일반] 처리 중: {state['query']}"}

def route(state: RouterState) -> Literal["weather", "math", "general"]:
    return state["category"]

builder = StateGraph(RouterState)
builder.add_node("classify", classify)
builder.add_node("weather", handle_weather)
builder.add_node("math", handle_math)
builder.add_node("general", handle_general)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route)
builder.add_edge("weather", END)
builder.add_edge("math", END)
builder.add_edge("general", END)

graph = builder.compile()

for query in ["오늘 날씨는?", "2+2를 계산해줘", "농담 하나 해줘"]:
    result = graph.invoke({"query": query}, config=lf_config)
    print(f"  {query} -> {result['result']}")

  오늘 날씨는? -> [날씨] 날씨 확인 중: 오늘 날씨는?
  2+2를 계산해줘 -> [수학] 계산 중: 2+2를 계산해줘
  농담 하나 해줘 -> [일반] 처리 중: 농담 하나 해줘


## 2.5 메시지 기반 상태

LLM 에이전트에 적합한 `MessagesState`를 사용합니다.

- `MessagesState`는 `messages: Annotated[list[AnyMessage], add_messages]`를 포함하는 사전 정의된 상태
- `add_messages` 리듀서가 메시지 리스트를 자동으로 누적
- LLM 응답을 메시지 히스토리에 자연스럽게 추가

In [6]:
from langgraph.graph import MessagesState
from langchain.messages import HumanMessage, SystemMessage

def chatbot(state: MessagesState) -> dict:
    response = model.invoke(state["messages"], config=lf_config)
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage(content="2+2는 얼마인가요?")]}, config=lf_config)
print("응답:", result["messages"][-1].content)

응답: 2+2는 4입니다.


## 2.6 입출력 스키마

그래프의 입출력을 내부 상태와 분리합니다.

- `StateGraph(InternalState, input_schema=InputSchema, output_schema=OutputSchema)`
- 입력 스키마: 외부에서 받는 데이터 만 포함
- 출력 스키마: 외부로 내보내는 데이터만 포함
- 내부 상태: 중간 처리용 필드 포함 (외부에 노출되지 않음)

In [7]:
from typing import TypedDict

class InputSchema(TypedDict):
    question: str


class OutputSchema(TypedDict):
    answer: str


class InternalState(TypedDict):
    question: str
    answer: str
    intermediate: str  # internal only


def process(state: InternalState) -> dict:
    return {
        "intermediate": state["question"].upper()
    }


def answer(state: InternalState) -> dict:
    return {
        "answer": f"'{state['intermediate']}'에 대한 답: 42"
    }


builder = StateGraph(
    InternalState,
    input_schema=InputSchema,
    output_schema=OutputSchema
)

builder.add_node("process", process)
builder.add_node("answer", answer)

builder.add_edge(START, "process")
builder.add_edge("process", "answer")
builder.add_edge("answer", END)

graph = builder.compile()

result = graph.invoke(
    {
        "question": "삶의 의미"
    },
    config=lf_config
)

print("출력:", result)

# intermediate 필드는 출력에 포함되지 않음

출력: {'answer': "'삶의 의미'에 대한 답: 42"}


## 2.7 요약

이번 노트북에서 배운 내용을 정리합니다.

| 개념 | 설명 |
|------|------|
| **StateGraph** | 상태 스키마 기반 그래프 빌더 |
| **Node** | Python 함수로 정의된 처리 단위 |
| **Edge** | 노드 간 고정 연결 (`add_edge`) |
| **Conditional Edge** | 상태 기반 동적 분기 (`add_conditional_edges`) |
| **Reducer** | `Annotated` + `operator.add`로 상태 누적 방식 정의 |
| **MessagesState** | LLM 대화용 사전 정의된 상태 |
| **Input/Output Schema** | 내부 상태와 외부 입출력 분리 |

### 다음 단계
→ **[03_functional_api.ipynb](./03_functional_api.ipynb)**: Functional API를 배웁니다.
